# Philadelphia CIP Parser

Parses all PDFs in `Philadelphia/PDF/` and writes one CSV per PDF into `Philadelphia/CSV/`.

**Key rule:** `year_YYYY` = the **total investment row** (a line of numbers with no funding-source suffix).  
Individual source rows (`CN`, `FT`, `XN`, `SB`, etc.) are summed implicitly in that row — they are **not** used directly.  
`project_total` = the **range-total column** (last column in the total row, e.g. "2021-2026" sum).  
`start_year` / `end_year` = first / last plan year with a non-zero investment.  
`project_justification` = same text as `project_description` (Philadelphia PDFs do not separate them).

## Format families detected
| Format | PDFs | Pages |
|--------|------|-------|
| **A – text** | 2019, 2020 | Project tables are selectable text; parsed with pdfplumber word positions |
| **B – image** | 2021–2025 | Project tables are scanned images; parsed with pytesseract OCR |

## Cell 1 – Imports & paths

In [19]:
import re
import warnings
from pathlib import Path
from collections import defaultdict

import pdfplumber
import pdf2image
import pytesseract
import pandas as pd

warnings.filterwarnings('ignore')

# ── Paths ───────────────────────────────────────────────────────────────────
# Notebook lives in Scripts/; data lives in the sibling Philadelphia/ folder.
SCRIPTS_DIR = Path('.').resolve()             # …/CIPBD/Scripts/
CITY_DIR    = SCRIPTS_DIR.parent / 'Philadelphia'
PDF_DIR     = CITY_DIR / 'PDF'
CSV_DIR     = CITY_DIR / 'CSV'
CSV_DIR.mkdir(exist_ok=True)

REQUIRED_COLS = [
    'cip_year', 'project_type', 'source_page', 'department',
    'project_name', 'project_id', 'address_location',
    'start_year', 'end_year',
    'project_description', 'project_justification',
    'previous_appropriations', 'project_total',
]

# All known Philadelphia funding-source codes
ALL_SOURCE_CODES = {
    'CN','CT','CR','CA','A',
    'XN','XT','XR',
    'Z',
    'FB','FO','FT',
    'PB','PT',
    'SB','SO','ST',
    'TB','TO','TT',
}

# Compiled patterns
SOURCE_CODE_RE = re.compile(r'^[A-Z]{2,3}$')
NUM_RE         = re.compile(r'^-?[\d,]+$')
YEAR_RE        = re.compile(r'^(\d{4})$')
PROJ_NUM_RE    = re.compile(r'^(\d+[A-Z]?\.?)\s+(.+)$')
DEPT_HDR_RE    = re.compile(r'^[A-Z][A-Z &\-/()]+$')

print('Paths OK.')
print(f'  PDF_DIR : {PDF_DIR}')
print(f'  CSV_DIR : {CSV_DIR}')
print('PDFs found:', sorted(p.name for p in PDF_DIR.glob('*.pdf')))

Paths OK.
  PDF_DIR : C:\Users\vince\Documents\GitHub\CIPBD\Philadelphia\PDF
  CSV_DIR : C:\Users\vince\Documents\GitHub\CIPBD\Philadelphia\CSV
PDFs found: ['2019.pdf', '2020.pdf', '2021.pdf', '2022.pdf', '2023.pdf', '2024.pdf', '2025.pdf']


## Cell 2 – Format detection

In [20]:
def classify_pdf(pdf_path):
    """
    Returns (overall_format, {page_num: 'text'|'image'|'other'}).
    A PDF is 'text' if the majority of its data pages (pg > 15)
    have extractable characters; otherwise 'image'.
    """
    page_fmt = {}
    with pdfplumber.open(pdf_path) as pdf:
        for i, pg in enumerate(pdf.pages):
            n_chars  = len(pg.chars)
            n_images = len(pg.images)
            if n_chars > 300:
                page_fmt[i + 1] = 'text'
            elif n_images > 0:
                page_fmt[i + 1] = 'image'
            else:
                page_fmt[i + 1] = 'other'

    data_pages = {pg: fmt for pg, fmt in page_fmt.items() if pg > 15}
    n_text  = sum(1 for f in data_pages.values() if f == 'text')
    n_image = sum(1 for f in data_pages.values() if f == 'image')
    overall = 'text' if n_text >= n_image else 'image'
    return overall, page_fmt


pdf_files  = sorted(PDF_DIR.glob('*.pdf'))
format_map = {}
for p in pdf_files:
    overall, per_page = classify_pdf(p)
    format_map[p.stem] = (overall, per_page)
    n_t = sum(1 for f in per_page.values() if f == 'text')
    n_i = sum(1 for f in per_page.values() if f == 'image')
    print(f'{p.name}: Format {overall.upper()}  (text pages: {n_t}, image pages: {n_i})')

2019.pdf: Format TEXT  (text pages: 207, image pages: 1)
2020.pdf: Format TEXT  (text pages: 208, image pages: 1)
2021.pdf: Format IMAGE  (text pages: 80, image pages: 184)
2022.pdf: Format IMAGE  (text pages: 94, image pages: 179)
2023.pdf: Format IMAGE  (text pages: 91, image pages: 191)
2024.pdf: Format IMAGE  (text pages: 87, image pages: 194)
2025.pdf: Format IMAGE  (text pages: 65, image pages: 214)


## Cell 3 – Shared utilities

In [21]:
# -- Number helpers
def parse_num(s):
    try:
        return float(str(s).replace(',', '').strip())
    except (ValueError, TypeError):
        return None

def is_source_code(tok):
    return tok in ALL_SOURCE_CODES

# Combined token: number immediately followed by source code, e.g. '300CN', '6,000CN'
COMBINED_SRC_RE = re.compile(r'^[\d,]+[A-Z]{2,3}$')

def has_source_tokens(row_words):
    """True if any right-side token is a pure source code OR a combined number+code."""
    return any(
        is_source_code(w['text']) or bool(COMBINED_SRC_RE.match(w['text']))
        for w in row_words if w['x0'] > 280
    )

# -- Word-row grouping
def group_words_by_row(words, y_tol=3):
    rows = defaultdict(list)
    for w in words:
        key = round(w['top'] / y_tol) * y_tol
        rows[key].append(w)
    for key in rows:
        rows[key].sort(key=lambda w: w['x0'])
    return rows

# -- Year-column detection from word positions
def detect_year_cols_from_words(words):
    """
    Scan rows for a header with >= 2 four-digit years.
    Uses setdefault to keep FIRST occurrence of each year,
    ignoring the repeated range-total column.
    """
    rows = group_words_by_row(words, y_tol=4)
    for _, row_words in sorted(rows.items()):
        years = [int(w['text']) for w in row_words
                 if YEAR_RE.match(w['text']) and 2018 <= int(w['text']) <= 2040]
        if len(years) >= 2:
            year_x = {}
            for w in row_words:
                if YEAR_RE.match(w['text']) and 2018 <= int(w['text']) <= 2040:
                    yr = int(w['text'])
                    cx = (w['x0'] + w['x1']) / 2
                    year_x.setdefault(yr, cx)
            return year_x
    return {}

def detect_year_cols_from_text(line):
    years = []
    for t in line.split():
        m = YEAR_RE.match(t)
        if m and 2018 <= int(m.group(1)) <= 2040:
            yr = int(m.group(1))
            if yr not in years:
                years.append(yr)
    return years

# -- Total-row detection
def is_total_row(row_words):
    """
    True when the right half (x0 > 280) contains ONLY plain numeric tokens.
    Rows with source codes OR combined '300CN' tokens are NOT total rows.
    """
    right = [w for w in row_words if w['x0'] > 280]
    if not right:
        return False
    return all(
        not is_source_code(w['text'])
        and not COMBINED_SRC_RE.match(w['text'])
        and bool(NUM_RE.match(w['text'].replace(',', '')))
        for w in right
    )

# -- Map a row's numbers to year columns (handles combined tokens too)
def map_row_to_years(row_words, year_x, x_tol=30):
    """
    Assign numeric values to the nearest year column.
    Handles plain numbers AND combined tokens like '300CN' (strips code suffix).
    Returns (yr_vals, range_total).
    """
    if not year_x:
        return {}, None
    sorted_years = sorted(year_x.items(), key=lambda kv: kv[1])
    result    = {}
    unmatched = []
    for w in [w for w in row_words if w['x0'] > 280]:
        text = w['text']
        if is_source_code(text):
            continue   # bare source code, no number
        if COMBINED_SRC_RE.match(text):
            num_str = re.sub(r'[A-Z]+$', '', text)
            val = parse_num(num_str)
        else:
            val = parse_num(text)
        if val is None:
            continue
        cx = (w['x0'] + w['x1']) / 2
        nearest_yr, nearest_x = min(sorted_years, key=lambda kv: abs(kv[1] - cx))
        if abs(nearest_x - cx) <= x_tol:
            result[nearest_yr] = result.get(nearest_yr, 0) + val
        else:
            unmatched.append(val)
    range_total = unmatched[-1] if unmatched else None
    return result, range_total

# -- Project-title detection
def categorise_row(row_words):
    left = [w for w in row_words if w['x0'] < 280]
    if not left:
        return ('other', '')
    text = ' '.join(w['text'] for w in left)
    if re.match(r'^(Totals|TOTALS)\s*[-\u2013]', text, re.I):
        return ('totals', text)
    m = PROJ_NUM_RE.match(text)
    if m:
        return ('project', m.group(1).rstrip('.'), m.group(2).strip())
    if DEPT_HDR_RE.match(text) and len(text) > 3:
        return ('dept_or_type', text)
    return ('other', text)

# -- Department abbreviation map
DEPT_ABBREV = {
    'ART MUSEUM': 'ART', 'AVIATION': 'AVN',
    'PHILADELPHIA INTERNATIONAL AIRPORT': 'AVN',
    'COMMERCE': 'COM', 'FINANCE': 'FIN', 'FIRE': 'FIRE',
    'FLEET': 'FLEET', 'FLEET SERVICES': 'FLEET',
    'FREE LIBRARY': 'LIB', 'HEALTH': 'HLTH',
    'HOMELESS SERVICES': 'HS', 'OFFICE OF HOMELESS SERVICES': 'HS',
    'HOUSING': 'HOUS', 'HUMAN SERVICES': 'DHS',
    'L&I': 'LI', 'LICENSES AND INSPECTIONS': 'LI',
    'MANAGING DIRECTOR': 'MD', 'MURAL ARTS': 'MURAL',
    'PARKS AND RECREATION': 'PPR', 'PLANNING': 'PLAN',
    'POLICE': 'PPD', 'PRISONS': 'PRISONS',
    'PUBLIC PROPERTY': 'PP', 'RECORDS': 'REC',
    'REVENUE': 'REV', 'SCHOOLS': 'SDP', 'SEPTA': 'SEPTA',
    'STREETS': 'ST', 'WATER': 'WATER',
    'WATER DEPARTMENT': 'WATER', 'ZOO': 'ZOO',
}

def dept_to_abbrev(dept):
    return DEPT_ABBREV.get(dept.upper().strip(), dept.strip()[:4])

def build_project_id(dept_abbrev, line_num):
    return f'{dept_abbrev}.{line_num}'

print('Utilities loaded.')


Utilities loaded.


## Cell 4 – Format A parser (text-based: 2019, 2020)

Uses pdfplumber word positions.  The **total row** (numbers only, no source codes)
is the anchor — individual source rows (`CN`, `FT`, `SB` …) are skipped for `year_YYYY`.  
The **range-total column** (rightmost number, beyond x_tol) is saved as `project_total`.

In [22]:
def _flush_pending(pending, src_yr_accum, desc_buf, projects):
    """
    Close out a pending project using accumulated source-row values.
    (Projects whose total row was already captured are appended inside
    the total-row branch; this function handles the source-only case.)
    """
    if pending is None:
        return None, {}
    has_yr = any(k.startswith('year_') and pending.get(k, '') != ''
                 for k in pending)
    if not has_yr and src_yr_accum:
        for yr, val in src_yr_accum.items():
            pending[f'year_{yr}'] = val
        pending['project_total'] = sum(src_yr_accum.values())
        desc_text = ' '.join(desc_buf).strip()
        pending['project_description']  = desc_text
        pending['project_justification'] = desc_text
        projects.append(pending)
    return None, {}


def parse_text_page(page, pg_num, cip_year, ctx):
    """
    Parse one pdfplumber page.

    Two paths to capture year values:
      Path A – Pure-number total row exists:
        If NO source rows have been accumulated yet, use the total row
        directly (most accurate: includes the PDF range-total column).
        If source rows ARE already accumulated, prefer them -- the
        apparent 'total row' is either a genuine duplicate of the source
        sum (safe to ignore) or a stray page number (must ignore).
      Path B – Source-only project (e.g. single funding source, no
        separate numeric total row):
        Accumulate source-row values in src_yr_accum and flush when
        the project ends (new project / section header / page end).
    """
    words = page.extract_words(x_tolerance=3, y_tolerance=3)
    if not words:
        return []

    year_x = detect_year_cols_from_words(words)
    if year_x:
        ctx['year_x'] = year_x
    else:
        year_x = ctx.get('year_x', {})
    if not year_x:
        return []

    rows         = group_words_by_row(words, y_tol=3)
    projects     = []
    pending      = None
    desc_buf     = []
    src_yr_accum = {}  # year values accumulated from source rows (fallback)

    for top in sorted(rows):
        rw = rows[top]

        # -- PURE-NUMBER TOTAL ROW ----------------------------------------
        if is_total_row(rw):
            if pending is not None:
                if src_yr_accum:
                    # Source values already on hand.  Use them.
                    # The numeric-only row is either a legitimate duplicate
                    # of the accumulated sum, or a stray page number.
                    # Either way, the accumulated values are correct.
                    pending, src_yr_accum = _flush_pending(
                        pending, src_yr_accum, desc_buf, projects)
                    desc_buf = []
                else:
                    # No source rows yet -- genuine total row.
                    yr_vals, range_total = map_row_to_years(rw, year_x)
                    if yr_vals:
                        for yr, val in yr_vals.items():
                            pending[f'year_{yr}'] = val
                        pending['project_total'] = (
                            range_total if range_total is not None
                            else sum(yr_vals.values())
                        )
                        desc_text = ' '.join(desc_buf).strip()
                        pending['project_description']  = desc_text
                        pending['project_justification'] = desc_text
                        projects.append(pending)
                    pending = None; src_yr_accum = {}; desc_buf = []
            continue

        # -- SOURCE ROW (source codes or combined tokens) -----------------
        if has_source_tokens(rw):
            if pending is not None:
                yr_vals, _ = map_row_to_years(rw, year_x)
                for yr, val in yr_vals.items():
                    src_yr_accum[yr] = src_yr_accum.get(yr, 0) + val
                left = ' '.join(w['text'] for w in rw if w['x0'] < 280).strip()
                if left:
                    desc_buf.append(left)
            continue

        # -- TEXT ROW -> classify -----------------------------------------
        cat = categorise_row(rw)

        if cat[0] == 'dept_or_type':
            pending, src_yr_accum = _flush_pending(
                pending, src_yr_accum, desc_buf, projects)
            desc_buf = []
            txt = cat[1]
            if txt in DEPT_ABBREV:
                ctx.update({'dept': txt, 'proj_type': '', 'main_proj_num': ''})
            elif 'TOTAL' not in txt:
                ctx['proj_type'] = txt

        elif cat[0] == 'project':
            num_part, name_part = cat[1], cat[2]
            if 'total' in name_part.lower():
                pending, src_yr_accum = _flush_pending(
                    pending, src_yr_accum, desc_buf, projects)
                desc_buf = []; continue
            # Flush previous project before starting the new one
            pending, src_yr_accum = _flush_pending(
                pending, src_yr_accum, desc_buf, projects)
            desc_buf = []

            dept        = ctx.get('dept', '')
            dept_abbrev = dept_to_abbrev(dept)
            x0_first    = rw[0]['x0'] if rw else 0
            is_subline  = (x0_first > 60 and re.match(r'^\d+$', num_part))
            if is_subline:
                main = ctx.get('main_proj_num', '')
                pid  = build_project_id(dept_abbrev,
                                        f'{main}.{num_part}' if main else num_part)
            else:
                if re.match(r'^\d+$', num_part):
                    ctx['main_proj_num'] = num_part
                pid = build_project_id(dept_abbrev, num_part)

            pending = {
                'cip_year': cip_year, 'project_type': ctx.get('proj_type', ''),
                'source_page': pg_num, 'department': dept,
                'project_name': name_part, 'project_id': pid,
                'address_location': '', 'start_year': '', 'end_year': '',
                'project_description': '', 'project_justification': '',
                'previous_appropriations': '', 'project_total': '',
            }

        elif cat[0] == 'totals':
            pending, src_yr_accum = _flush_pending(
                pending, src_yr_accum, desc_buf, projects)
            desc_buf = []

        else:
            left = ' '.join(w['text'] for w in rw if w['x0'] < 280).strip()
            if left and pending is not None:
                desc_buf.append(left)

    # End of page: flush any project still pending
    _flush_pending(pending, src_yr_accum, desc_buf, projects)
    return projects


def parse_format_a(pdf_path, page_fmts):
    """Parse a text-based PDF.  Returns (cip_year, plan_years, projects)."""
    ctx = {'dept': '', 'proj_type': '', 'main_proj_num': '', 'year_x': {}}
    all_projects = []
    cip_year, plan_years = None, []

    with pdfplumber.open(pdf_path) as pdf:
        for pg_idx, pg in enumerate(pdf.pages):
            pg_num = pg_idx + 1
            if page_fmts.get(pg_num) != 'text':
                continue
            txt = pg.extract_text() or ''
            if cip_year is None:
                m = re.search(r'FY\s*(\d{4})\s*[-\u2013]\s*(\d{4})', txt)
                if m:
                    cip_year = int(m.group(1))
            if not plan_years:
                for line in txt.split('\n')[:5]:
                    yrs = detect_year_cols_from_text(line)
                    if len(yrs) >= 2:
                        plan_years = yrs; break
            if 'x000' not in txt.lower():
                continue
            all_projects.extend(parse_text_page(pg, pg_num, cip_year, ctx))

    return cip_year, plan_years, all_projects


print('Format A parser ready.')


Format A parser ready.


## Cell 5 – Format B parser (image-based: 2021–2025, OCR)

Renders each image page with pdf2image and reads it with pytesseract.  
The same total-row logic applies; the last number (range total) is saved as `project_total`.

In [23]:
TESS_CONFIG = '--psm 6 --oem 1'


def ocr_page(pdf_path, pg_1indexed, dpi=300):
    """Render one page to a grayscale image and return OCR text."""
    imgs = pdf2image.convert_from_path(
        str(pdf_path), dpi=dpi,
        first_page=pg_1indexed, last_page=pg_1indexed,
        grayscale=True,
    )
    return pytesseract.image_to_string(imgs[0], config=TESS_CONFIG) if imgs else ''


def parse_token_line_b(line, plan_years):
    """
    Classify one OCR text line.  Returns a tuple whose first element is:
      'header'     – year-column header
      'total'      – total row  → (yr_vals dict, range_total)
      'source'     – source row → skip
      'project'    – project title
      'dept_type'  – dept / project-type header
      'totals_hdr' – section subtotals header
      'other'
    """
    line = line.strip()
    if not line or re.match(r'^\d{1,3}$', line):  # bare page number
        return ('other',)
    if re.search(r'x000|\$x', line, re.I):
        return ('other',)
    if re.match(r'^(Totals|TOTALS)\s*[-–]', line, re.I):
        return ('totals_hdr',)

    tokens = line.split()

    # Header line with ≥2 4-digit years?
    yrs = [int(t) for t in tokens if YEAR_RE.match(t) and 2018 <= int(t) <= 2040]
    if len(yrs) >= 2:
        return ('header', sorted(set(yrs)))

    # Find where the numeric block starts
    nb_start = None
    for i, tok in enumerate(tokens):
        if NUM_RE.match(tok.replace(',', '')) or is_source_code(tok):
            nb_start = i; break

    if nb_start is None:
        m = PROJ_NUM_RE.match(line)
        if m:
            return ('project', m.group(1).rstrip('.'), m.group(2).strip())
        if DEPT_HDR_RE.match(line) and len(line) > 3:
            return ('dept_type', line)
        return ('other',)

    right  = tokens[nb_start:]
    has_sc = any(is_source_code(t) for t in right)

    # Collect all numeric values (left-to-right)
    nums = [float(t.replace(',', '')) for t in right
            if NUM_RE.match(t.replace(',', ''))]
    if not nums:
        return ('other',)

    if has_sc:
        return ('source',)   # individual source row – ignore for year values

    # Map position → plan year; last number is the range total
    yr_vals     = {plan_years[j]: v for j, v in enumerate(nums[:-1]) if j < len(plan_years)}
    range_total = nums[-1] if len(nums) > 1 else None
    return ('total', yr_vals, range_total)


def parse_ocr_text(ocr_text, pg_num, cip_year, ctx, plan_years):
    """Parse OCR output from one image page.  Returns list of project dicts."""
    projects = []
    pending  = None
    desc_buf = []

    for line in ocr_text.split('\n'):
        cat = parse_token_line_b(line, plan_years)

        if cat[0] == 'header':
            # Deduplicate: remove repeated entries (range-total re-uses start year)
            uniq = []
            seen = set()
            for y in cat[1]:
                if y not in seen:
                    uniq.append(y); seen.add(y)
            plan_years = uniq[:6]
            ctx['plan_years'] = plan_years

        elif cat[0] == 'total':
            yr_vals     = cat[1]
            range_total = cat[2] if len(cat) > 2 else None
            if pending is not None and yr_vals:
                for yr, val in yr_vals.items():
                    pending[f'year_{yr}'] = val
                pending['project_total'] = (
                    range_total if range_total is not None
                    else sum(yr_vals.values())
                )
                desc_text = ' '.join(desc_buf).strip()
                pending['project_description']  = desc_text
                pending['project_justification'] = desc_text
                projects.append(pending)
            pending  = None
            desc_buf = []

        elif cat[0] == 'source':
            pass   # deliberately ignored

        elif cat[0] == 'project':
            num_part, name_part = cat[1], cat[2]
            if 'total' in name_part.lower():
                pending = None; desc_buf = []; continue
            dept        = ctx.get('dept', '')
            dept_abbrev = dept_to_abbrev(dept)
            is_sub      = bool(re.match(r'^\d+$', num_part))
            if is_sub:
                main = ctx.get('main_proj_num', '')
                pid  = build_project_id(dept_abbrev,
                                        f'{main}.{num_part}' if main else num_part)
            else:
                if re.match(r'^\d+$', num_part):
                    ctx['main_proj_num'] = num_part
                pid = build_project_id(dept_abbrev, num_part)
            pending = {
                'cip_year': cip_year, 'project_type': ctx.get('proj_type', ''),
                'source_page': pg_num, 'department': dept,
                'project_name': name_part, 'project_id': pid,
                'address_location': '', 'start_year': '', 'end_year': '',
                'project_description': '', 'project_justification': '',
                'previous_appropriations': '', 'project_total': '',
            }
            desc_buf = []

        elif cat[0] == 'dept_type':
            txt = cat[1]
            if txt.upper() in DEPT_ABBREV:
                ctx.update({'dept': txt.upper(), 'proj_type': '', 'main_proj_num': ''})
            elif 'TOTAL' not in txt.upper():
                ctx['proj_type'] = txt
            pending  = None
            desc_buf = []

        elif cat[0] in ('totals_hdr', 'other'):
            if cat[0] == 'other' and len(cat) > 1 and cat[1] and pending:
                desc_buf.append(str(cat[1]))
            elif cat[0] == 'totals_hdr':
                pending = None; desc_buf = []

    return projects


def parse_format_b(pdf_path, page_fmts):
    """Parse an image-based PDF via OCR.  Returns (cip_year, plan_years, projects)."""
    ctx = {'dept': '', 'proj_type': '', 'main_proj_num': '', 'plan_years': []}
    all_projects = []
    cip_year, plan_years = None, []

    # Extract CIP year from text-rich intro pages
    with pdfplumber.open(pdf_path) as pdf:
        for pg in pdf.pages[:30]:
            txt = pg.extract_text() or ''
            m = re.search(r'(?:FY|FISCAL YEARS?)\s*(\d{4})\s*[-–]\s*(\d{4})', txt, re.I)
            if m:
                cip_year   = int(m.group(1))
                plan_years = list(range(cip_year, cip_year + 6))
                ctx['plan_years'] = plan_years
                break

    if cip_year is None:   # fallback: derive from filename
        pub_year   = int(pdf_path.stem) if pdf_path.stem.isdigit() else 2020
        cip_year   = pub_year + 1
        plan_years = list(range(cip_year, cip_year + 6))
        ctx['plan_years'] = plan_years

    print(f'  cip_year={cip_year}, plan_years={plan_years}')

    image_pages = sorted(pg for pg, fmt in page_fmts.items()
                         if fmt == 'image' and pg >= 15)
    total = len(image_pages)

    for i, pg_num in enumerate(image_pages, 1):
        if i % 20 == 0:
            print(f'  OCR: page {pg_num} ({i}/{total})…')
        try:
            ocr_text = ocr_page(pdf_path, pg_num)
        except Exception as exc:
            print(f'  OCR error page {pg_num}: {exc}')
            continue
        rows = parse_ocr_text(ocr_text, pg_num, cip_year, ctx,
                              ctx.get('plan_years', plan_years))
        all_projects.extend(rows)

    return cip_year, plan_years, all_projects


print('Format B parser ready.')

Format B parser ready.


## Cell 6 – Main loop: parse all PDFs → write CSVs

In [24]:
def build_dataframe(projects, plan_years):
    """
    Convert project list to DataFrame with correct column order.

    Derived fields (computed here, not in the parsers):
      start_year  – first plan-year column with a non-zero value
      end_year    – last  plan-year column with a non-zero value
    project_total is already set by the parsers from the PDF's range-total column;
    this function falls back to summing year columns only if it is missing.
    """
    if not projects:
        return pd.DataFrame(columns=REQUIRED_COLS)

    df = pd.DataFrame(projects)
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = ''

    year_cols = [f'year_{y}' for y in sorted(plan_years)]
    for col in year_cols:
        if col not in df.columns:
            df[col] = ''

    # ── start_year / end_year ─────────────────────────────────────────────
    def get_start_year(row):
        for c in year_cols:
            try:
                if float(row.get(c, '') or 0) > 0:
                    return int(c.replace('year_', ''))
            except (ValueError, TypeError):
                pass
        return ''

    def get_end_year(row):
        last = ''
        for c in year_cols:
            try:
                if float(row.get(c, '') or 0) > 0:
                    last = int(c.replace('year_', ''))
            except (ValueError, TypeError):
                pass
        return last

    df['start_year'] = df.apply(get_start_year, axis=1)
    df['end_year']   = df.apply(get_end_year,   axis=1)

    # ── project_total fallback ────────────────────────────────────────────
    # Parsers already set project_total from the PDF's range-total column.
    # For any rows where it is still missing, sum the year columns.
    def fallback_total(row):
        existing = row.get('project_total', '')
        try:
            if existing != '' and float(existing) > 0:
                return existing   # already set by parser
        except (ValueError, TypeError):
            pass
        vals = []
        for c in year_cols:
            try:
                v = float(row.get(c, '') or '')
                if v != 0:
                    vals.append(v)
            except (ValueError, TypeError):
                pass
        return sum(vals) if vals else ''

    df['project_total'] = df.apply(fallback_total, axis=1)

    col_order = REQUIRED_COLS + year_cols
    extra     = [c for c in df.columns if c not in col_order]
    return df[col_order + extra]


summary = []

for pdf_path in sorted(PDF_DIR.glob('*.pdf')):
    stem          = pdf_path.stem
    overall, pfmt = format_map[stem]
    print(f'\n══ {pdf_path.name}  [Format {overall.upper()}] ══')

    if overall == 'text':
        cip_year, plan_years, projects = parse_format_a(pdf_path, pfmt)
    else:
        cip_year, plan_years, projects = parse_format_b(pdf_path, pfmt)

    # cip_year = the PDF filename (e.g. 2019), not the fiscal year
    # derived from the document header (which would be 2020 for 2019.pdf).
    file_year = int(stem) if stem.isdigit() else cip_year
    for p in projects:
        p['cip_year'] = file_year
    cip_year = file_year

    df       = build_dataframe(projects, plan_years)
    out_path = CSV_DIR / f'{stem}.csv'
    df.to_csv(out_path, index=False)

    print(f'  plan_years={plan_years}')
    print(f'  rows extracted: {len(df)}')
    print(f'  → {out_path}')
    summary.append({'file': pdf_path.name, 'fmt': overall,
                    'cip_year': cip_year, 'rows': len(df)})

print('\n\n══ ALL DONE ══')


══ 2019.pdf  [Format TEXT] ══
  plan_years=[2020, 2021, 2022, 2023, 2024, 2025]
  rows extracted: 452
  → C:\Users\vince\Documents\GitHub\CIPBD\Philadelphia\CSV\2019.csv

══ 2020.pdf  [Format TEXT] ══
  plan_years=[2021, 2022, 2023, 2024, 2025, 2026]
  rows extracted: 442
  → C:\Users\vince\Documents\GitHub\CIPBD\Philadelphia\CSV\2020.csv

══ 2021.pdf  [Format IMAGE] ══
  cip_year=2022, plan_years=[2022, 2023, 2024, 2025, 2026, 2027]
  OCR error page 17: Unable to get page count. Is poppler installed and in PATH?
  OCR error page 41: Unable to get page count. Is poppler installed and in PATH?
  OCR error page 42: Unable to get page count. Is poppler installed and in PATH?
  OCR error page 43: Unable to get page count. Is poppler installed and in PATH?
  OCR error page 46: Unable to get page count. Is poppler installed and in PATH?
  OCR error page 47: Unable to get page count. Is poppler installed and in PATH?
  OCR error page 48: Unable to get page count. Is poppler installed and in 

## Cell 7 – Null-rate diagnostics

In [25]:
NULL_MARKERS = {'', 'nan', 'na', 'n/a', 'tbd', 'none'}

DIAG_COLS = ['start_year', 'end_year', 'project_total',
             'project_description', 'project_justification', 'department']

print(f'{"File":<20} {"Rows":>5}  ' + '  '.join(f'{c[:12]:>12}' for c in DIAG_COLS))
print('-' * (28 + 14 * len(DIAG_COLS)))

for info in summary:
    df  = pd.read_csv(CSV_DIR / info['file'].replace('.pdf', '.csv'), dtype=str).fillna('')
    n   = len(df)
    pct = lambda col: (
        f'{(~df[col].str.strip().str.lower().isin(NULL_MARKERS)).sum()}/{n}'
        if col in df else 'N/A'
    )
    vals = '  '.join(f'{pct(c):>12}' for c in DIAG_COLS)
    print(f'{info["file"].replace(".pdf",".csv"):<20} {n:>5}  {vals}')

File                  Rows    start_year      end_year  project_tota  project_desc  project_just    department
----------------------------------------------------------------------------------------------------------------
2019.csv               452       452/452       452/452       452/452       452/452       452/452       452/452
2020.csv               442       442/442       442/442       442/442       442/442       442/442       442/442
2021.csv                 0           0/0           0/0           0/0           0/0           0/0           0/0
2022.csv                 0           0/0           0/0           0/0           0/0           0/0           0/0
2023.csv                 0           0/0           0/0           0/0           0/0           0/0           0/0
2024.csv                 0           0/0           0/0           0/0           0/0           0/0           0/0
2025.csv                 0           0/0           0/0           0/0           0/0           0/0           0/0

## Cell 8 – Spot-check: verify the key fix

Confirms that `year_YYYY` equals the **total row** value, not a single source row.
Also verifies `start_year`, `end_year`, and `project_total` are populated.

Key case: **North Delaware River Waterfront-FY17** in `2019.pdf`  
PDF shows: 2,470 FT + 350 PT + 600 ST → **total row = 3,420**  
Range total column = 3,420 (only in year_2020, so project_total = 3,420, start_year = end_year = 2020).

In [26]:
CHECK_CASES = [
    # (csv_stem, name_fragment,                           field,          expected)
    ('2019', 'North Delaware River Waterfront-FY17',     'year_2020',     3420.0),
    ('2019', 'North Delaware River Waterfront-FY17',     'project_total', 3420.0),
    ('2019', 'North Delaware River Waterfront-FY17',     'start_year',    2020),
    ('2019', 'North Delaware River Waterfront-FY17',     'end_year',      2020),
    ('2020', 'North Delaware River Waterfront-FY17',     'year_2021',     3420.0),
    ('2020', 'North Delaware River Waterfront-FY17',     'project_total', 3420.0),
    ('2020', 'Airfield Area',                            'year_2021',    96950.0),
    ('2020', 'Airfield Area',                            'year_2026',   105000.0),
    ('2019', 'Airfield Area',                            'year_2020',   104900.0),
    ('2019', 'Airfield Area',                            'project_total', 628633.0),
    ('2019', 'Airfield Area',                            'start_year',    2020),
    ('2019', 'Airfield Area',                            'end_year',      2025),
    ('2019', 'Schuylkill River Waterfront',              'year_2020',    22000.0),
]

print(f'{"CSV":<8} {"Project (partial)":<44} {"Field":<16} '
      f'{"Expected":>10} {"Got":>12}  Status')
print('-' * 105)

all_ok = True
for stem, frag, field, expected in CHECK_CASES:
    csv_path = CSV_DIR / f'{stem}.csv'
    if not csv_path.exists():
        print(f'{stem:<8} {frag:<44} – file not found'); all_ok = False; continue

    df  = pd.read_csv(csv_path, dtype=str).fillna('')
    hit = df[df['project_name'].str.contains(frag, na=False, case=False)]

    if hit.empty:
        print(f'{stem:<8} {frag:<44} – NOT FOUND in CSV'); all_ok = False; continue

    raw = hit.iloc[0].get(field, '')
    try:
        got = float(raw) if '.' in str(raw) or (str(raw).lstrip('-').isdigit() and isinstance(expected, float)) else int(raw)
    except (ValueError, TypeError):
        got = raw

    status = '✓ PASS' if got == expected else '✗ FAIL'
    if '✗' in status:
        all_ok = False
    print(f'{stem:<8} {frag:<44} {field:<16} '
          f'{str(expected):>10} {str(got):>12}  {status}')

print()
if all_ok:
    print('All spot-checks passed ✓')
else:
    print('Some spot-checks failed — review parser output and re-run.')

CSV      Project (partial)                            Field              Expected          Got  Status
---------------------------------------------------------------------------------------------------------
2019     North Delaware River Waterfront-FY17         year_2020            3420.0       3420.0  ✓ PASS
2019     North Delaware River Waterfront-FY17         project_total        3420.0       3420.0  ✓ PASS
2019     North Delaware River Waterfront-FY17         start_year             2020         2020  ✓ PASS
2019     North Delaware River Waterfront-FY17         end_year               2020         2020  ✓ PASS
2020     North Delaware River Waterfront-FY17         – NOT FOUND in CSV
2020     North Delaware River Waterfront-FY17         – NOT FOUND in CSV
2020     Airfield Area                                year_2021           96950.0      96950.0  ✓ PASS
2020     Airfield Area                                year_2026          105000.0     105000.0  ✓ PASS
2019     Airfield Area     